In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


## Construindo uma aplicação de Q&A usando Amazon Bedrock Knowledge Bases - API Retrieve

### Contexto

Neste notebook, vamos explorar em profundidade a construção de uma aplicação de Q&A usando a API Retrieve das Amazon Bedrock Knowledge Bases. Aqui, vamos consultar a knowledge base para obter o número desejado de chunks de documento com base em busca por similaridade. Em seguida, vamos aumentar (augment) o prompt com os documentos relevantes e a query, que serão passados como entrada para o Anthropic Claude V2 para gerar a resposta.

Com uma knowledge base, você pode conectar de forma segura foundation models (FMs) do Amazon Bedrock aos dados da sua empresa para Retrieval Augmented Generation (RAG). O acesso a dados adicionais ajuda o modelo a gerar respostas mais relevantes, específicas ao contexto e precisas, sem a necessidade de retreinar continuamente o FM. Todas as informações recuperadas das knowledge bases vêm com atribuição de fonte (source attribution) para melhorar a transparência e minimizar alucinações. Para mais informações sobre como criar uma knowledge base usando o console, consulte este [post](https://docs.aws.amazon.com/bedrock/latest/userguide/knowledge-base.html).
Vamos cobrir 2 partes neste notebook:
- Na Parte 1, vamos mostrar como você pode usar a `RetrieveAPI` com foundation models do Amazon Bedrock. Vamos usar o modelo ``BEDROCK_TEXT_MODEL_ID``.
- Na Parte 2, vamos demonstrar a integração com o langchain.

### Padrão (Pattern)

Podemos implementar a solução usando o padrão Retrieval Augmented Generation (RAG). O RAG recupera dados de fora do language model (não paramétrico) e aumenta os prompts adicionando os dados relevantes recuperados como contexto. Aqui, estamos realizando RAG de forma eficaz sobre a knowledge base criada via console/sdk.

### Pré-requisito

Antes de conseguir responder às perguntas, os documentos precisam ser processados e armazenados em uma knowledge base. Para este notebook, usamos um `dataset sintético de relatórios financeiros 10K` para criar as Amazon Bedrock Knowledge Bases.

1. Faça upload dos seus documentos (data source) para um bucket Amazon S3.
2. Amazon Bedrock Knowledge Bases usando [01_create_ingest_documents_test_kb_multi_ds.ipynb](/knowledge-bases/01-rag-concepts/01_create_ingest_documents_test_kb_multi_ds.ipynb)
3. Anote o Knowledge Base ID


<!-- ![data_ingestion](./images/data_ingestion.png) -->
<img src="./images/data_ingestion.png" width=50% height=20% />


#### Percurso do notebook



Neste notebook vamos usar a `Retrieve API` fornecida pelas Amazon Bedrock Knowledge Bases, que converte as queries do usuário em embeddings, busca na knowledge base e retorna os resultados relevantes, dando a você mais controle para construir workflows customizados em cima dos resultados da busca semântica. A saída da `Retrieve API` inclui os `retrieved text chunks`, o `location type` e o `URI` dos dados de origem, assim como os `scores` de relevância das recuperações.


Em seguida, vamos usar os text chunks gerados e aumentar (augment) o prompt original com eles, passando-os pelo modelo ``BEDROCK_TEXT_MODEL_ID`` usando padrões de prompt engineering de acordo com o seu caso de uso.


### CASO DE USO:

#### Dataset

Neste exemplo, você vai usar os relatórios financeiros 10k da Octank (dataset gerado sinteticamente) como corpus de texto para realizar Q&A. Esses dados já foram ingeridos nas Amazon Bedrock Knowledge Bases. Você vai precisar do `knowledge base id` para executar este exemplo.
No seu caso de uso específico, você pode sincronizar arquivos diferentes para diferentes tópicos de domínio e consultar este notebook da mesma forma para avaliar as respostas do modelo usando a retrieve API das knowledge bases.


### Python 3.10

⚠  Para este laboratório precisamos executar o notebook com base em um runtime Python 3.10. ⚠

Se você estiver realizando o workshop a partir do seu ambiente local, fora do Amazon SageMaker Studio, certifique-se de estar executando um runtime Python > 3.10.

### Setup

Para executar este notebook você precisará instalar os seguintes pacotes.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

#### Reinicie o kernel com os pacotes atualizados que foram instalados através das dependências acima

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
from IPython import get_ipython

workshop_state = None
ip = get_ipython()
if ip is None:
    raise RuntimeError("This notebook must run inside a Jupyter/IPython kernel.")
if "store" not in ip.magics_manager.magics["line"]:
    ip.run_line_magic("load_ext", "storemagic")
workshop_state = load_workshop_state()

if not isinstance(workshop_state, dict) or workshop_state.get("schema_version") != 1:
    raise RuntimeError(
        "Shared workshop state is missing or obsolete. Run "
        "01_create_ingest_documents_test_kb_multi_ds.ipynb through its "
        "state-persistence cell, then rerun this notebook."
    )

required_state_keys = ("kb_id", "knowledge_base_name", "region", "account_id")
missing_state_keys = [key for key in required_state_keys if not workshop_state.get(key)]
if missing_state_keys:
    raise RuntimeError(
        "Shared workshop state is incomplete; missing: "
        + ", ".join(missing_state_keys)
        + ". Re-run the creator notebook and persist its state."
    )

kb_id = workshop_state["kb_id"]

# AWS validation runs after the clients are initialized in the next setup cell.


### Siga os passos abaixo para iniciar o client do bedrock:

1. Importe as bibliotecas necessárias, incluindo o langchain para seleção do modelo bedrock, e o llama index para armazenar o service context contendo as instâncias do llm e do embedding model. Vamos usar esse service context mais adiante no notebook para avaliar as respostas da nossa aplicação de Q&A.

2. Inicialize o ``BEDROCK_TEXT_MODEL_ID`` como nosso large language model para realizar as query completions usando o padrão RAG com a knowledge base fornecida, uma vez que tenhamos todas as buscas de text chunk através da API `retrieve`.

In [ ]:
import boto3
import pprint
from botocore.client import Config
from botocore.exceptions import ClientError
import json

pp = pprint.PrettyPrinter(indent=2)
session = boto3.session.Session()
region = session.region_name
account_id = boto3.client('sts').get_caller_identity()["Account"]
bedrock_config = Config(connect_timeout=120, read_timeout=120, retries={'max_attempts': 0})
bedrock_client = boto3.client('bedrock-runtime', region_name=region)
bedrock_agent_client = boto3.client("bedrock-agent-runtime",
                              config=bedrock_config, region_name=region)
bedrock_agent_control_client = boto3.client("bedrock-agent",
                              config=bedrock_config, region_name=region)
print(region)

region_name = region
expected_region = workshop_state["region"]
expected_account_id = workshop_state["account_id"]
if expected_region != region_name:
    raise RuntimeError(
        f"Shared workshop state belongs to region {expected_region}, but this notebook "
        f"is using {region_name}. Set the AWS region to {expected_region} or recreate the state."
    )
if expected_account_id != account_id:
    raise RuntimeError(
        f"Shared workshop state belongs to account {expected_account_id}, but the active "
        f"credentials use account {account_id}. Select the matching AWS credentials."
    )

try:
    kb_response = bedrock_agent_control_client.get_knowledge_base(knowledgeBaseId=kb_id)
except ClientError as exc:
    error_code = exc.response.get("Error", {}).get("Code")
    if error_code == "ResourceNotFoundException":
        raise RuntimeError(
            f"Knowledge Base {kb_id!r} from workshop_state does not exist in AWS "
            f"(account {account_id}, region {region_name}). Re-run notebook 01 and "
            "persist a fresh state before running this notebook."
        ) from exc
    raise RuntimeError(
        f"Could not validate Knowledge Base {kb_id!r} in AWS: "
        f"{error_code or type(exc).__name__}: {exc}"
    ) from exc

kb_summary = kb_response["knowledgeBase"]
expected_name = workshop_state["knowledge_base_name"]
if kb_summary.get("name") != expected_name:
    raise RuntimeError(
        f"Shared workshop state maps {kb_id!r} to {expected_name!r}, but AWS returned "
        f"{kb_summary.get('name')!r}. Re-run notebook 01 and persist a fresh state."
    )

print(
    f"Validated Knowledge Base {kb_id} ({kb_summary.get('status')}) "
    f"in account {account_id}, region {region_name}."
)


### Parte 1 - Retrieve API com foundation models do Amazon Bedrock

Defina uma função retrieve que chama a `Retrieve API` fornecida pelas Amazon Bedrock Knowledge Bases, que converte as queries do usuário em embeddings, busca na knowledge base e retorna os resultados relevantes, dando a você mais controle para construir workflows customizados em cima dos resultados da busca semântica. A saída da `Retrieve API` inclui os `retrieved text chunks`, o `location type` e o `URI` dos dados de origem, assim como os `scores` de relevância das recuperações. Você também pode usar a opção `overrideSearchType` em `retrievalConfiguration`, que oferece a escolha de usar `HYBRID` ou `SEMANTIC`. Por padrão, o serviço seleciona a estratégia correta para você, buscando dar os resultados mais relevantes, e se você quiser sobrescrever a opção padrão para usar busca híbrida ou semântica, pode definir o valor como `HYBRID/SEMANTIC`.

<!-- ![retrieveAPI](./images/retrieveAPI.png) -->
<img src="./images/retrieveAPI.png" width=50% height=20% />


In [ ]:
def retrieve(query, kbId, numberOfResults=5):
    return bedrock_agent_client.retrieve(
        retrievalQuery= {
            'text': query
        },
        knowledgeBaseId=kbId,
        retrievalConfiguration= {
            'vectorSearchConfiguration': {
                'numberOfResults': numberOfResults,
                'overrideSearchType': "HYBRID", # optional
            }
        }
    )

#### Inicialize o seu Knowledge base id antes de consultar respostas do LLM inicializado

A seguir, vamos chamar a `retrieve API`, passando `knowledge base id`, `number of results` e `query` como parâmetros.

`score`: Você pode visualizar o score associado a cada um dos text chunks retornados, que representa sua correlação com a query em termos de quão próxima é a correspondência.

In [ ]:
query = "What was the total operating lease liabilities and total sublease income of the Octank as of December 31, 2022?"
response = retrieve(query, kb_id, 5)
retrievalResults = response['retrievalResults']
pp.pprint(retrievalResults)

### Extrair os text chunks da resposta do retrieveAPI

Na célula abaixo, vamos buscar o contexto a partir dos resultados de retrieval.

In [ ]:
# fetch context from the response
def get_contexts(retrievalResults):
    contexts = []
    for retrievedResult in retrievalResults: 
        contexts.append(retrievedResult['content']['text'])
    return contexts

In [ ]:
contexts = get_contexts(retrievalResults)
pp.pprint(contexts)

### Prompt específico do modelo para personalizar as respostas

Aqui, vamos usar o prompt específico abaixo para que o modelo atue como um sistema de IA consultor financeiro que fornecerá respostas às perguntas usando informações baseadas em fatos e estatísticas sempre que possível. Vamos fornecer as respostas da `Retrieve API` acima como parte do `{contexts}` no prompt, para o modelo consultar, junto com a `query` do usuário.

In [ ]:
prompt = f"""
Human: You are a financial advisor AI system, and provides answers to questions by using fact based and statistical information when possible. 
Use the following pieces of information to provide a concise answer to the question enclosed in <question> tags. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
<context>
{contexts}
</context>

<question>
{query}
</question>

The response should be specific and use statistics or numbers when possible.

Assistant:"""

### Invocar o foundation model do Amazon Bedrock
Neste exemplo, vamos usar o foundation model ``BEDROCK_TEXT_MODEL_ID`` do Amazon Bedrock.
- Ele oferece máxima utilidade a um preço menor que os concorrentes, sendo projetado para ser o workhorse confiável e de alta resistência para implantações de IA em escala. O modelo `us.anthropic.claude-haiku-4-5-20251001-v1:0` consegue processar imagens e retornar saídas em texto, e apresenta uma janela de contexto de 200K.
- Atributos do modelo
    - Imagem para texto e código, conversação multilíngue, raciocínio e análise complexos

In [ ]:
# payload with model paramters
messages=[{ "role":'user', "content":[{'type':'text','text': prompt.format(contexts, query)}]}]
sonnet_payload = json.dumps({
    "anthropic_version": "bedrock-2023-05-31",
    "max_tokens": 512,
    "messages": messages,
    "temperature": 0.5,
        }  )

In [ ]:
import os

modelId = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")
accept = 'application/json'
contentType = 'application/json'
response = bedrock_client.invoke_model(body=sonnet_payload, modelId=modelId, accept=accept, contentType=contentType)
response_body = json.loads(response.get('body').read())
response_text = response_body.get('content')[0]['text']

pp.pprint(response_text)

## Parte 2 - Integração com LangChain
Neste notebook, vamos explorar em profundidade a construção de uma aplicação de Q&A usando a Retrieve API fornecida pelas Amazon Bedrock Knowledge Bases e o LangChain. Vamos consultar a knowledge base para obter o número desejado de chunks de documento com base em busca por similaridade, integrar isso com o retriever do LangChain e usar o modelo `us.anthropic.claude-haiku-4-5-20251001-v1:0` para responder às perguntas.

In [ ]:
# from langchain.llms.bedrock import Bedrock
import langchain
from langchain_aws import ChatBedrock
from langchain.retrievers.bedrock import AmazonKnowledgeBasesRetriever

llm = ChatBedrock(model_id=modelId, 
                  client=bedrock_client)

Crie um objeto `AmazonKnowledgeBasesRetriever` do LangChain, que vai chamar a `Retrieve API` fornecida pelas Amazon Bedrock Knowledge Bases, que converte as queries do usuário em embeddings, busca na knowledge base e retorna os resultados relevantes, dando a você mais controle para construir workflows customizados em cima dos resultados da busca semântica. A saída da `Retrieve API` inclui os `retrieved text chunks`, o `location type` e o `URI` dos dados de origem, assim como os `scores` de relevância das recuperações.

In [ ]:
query = "What was the total operating lease liabilities and total sublease income of the Octank as of December 31, 2022?"
retriever = AmazonKnowledgeBasesRetriever(
        knowledge_base_id=kb_id,
        retrieval_config={"vectorSearchConfiguration": 
                          {"numberOfResults": 4,
                           'overrideSearchType': "SEMANTIC", # optional
                           }
                          },
        # endpoint_url=endpoint_url,
        # region_name=region,
        # credentials_profile_name="<profile_name>",
    )
docs = retriever.get_relevant_documents(
        query=query
    )
pp.pprint(docs)

## Prompt específico do modelo para personalizar as respostas
Aqui, vamos usar o prompt específico abaixo para que o modelo atue como um sistema de IA consultor financeiro que fornecerá respostas às perguntas usando informações baseadas em fatos e estatísticas sempre que possível. Vamos fornecer as respostas da Retrieve API acima como parte do `{context}` no prompt, para o modelo consultar, junto com a `query` do usuário.

In [ ]:
from langchain.prompts import PromptTemplate

PROMPT_TEMPLATE = """
Human: You are a financial advisor AI system, and provides answers to questions by using fact based and statistical information when possible. 
Use the following pieces of information to provide a concise answer to the question enclosed in <question> tags. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
<context>
{context}
</context>

<question>
{question}
</question>

The response should be specific and use statistics or numbers when possible.

Assistant:"""
claude_prompt = PromptTemplate(template=PROMPT_TEMPLATE, 
                               input_variables=["context","question"])

### Integrando o retriever e o LLM definidos acima com a RetrievalQA Chain para construir a aplicação de Q&A.

In [ ]:
from langchain.chains import RetrievalQA

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": claude_prompt}
)

In [ ]:
answer = qa.invoke(query)
pp.pprint(answer)

## Conclusão
Você pode usar a Retrieve API para customizar sua aplicação baseada em RAG, usando a API `InvokeModel` do Bedrock, ou integrando com o LangChain usando o `AmazonKnowledgeBaseRetriever`.
A Retrieve API oferece a flexibilidade de usar qualquer foundation model fornecido pelo Amazon Bedrock, e escolher o tipo de busca certo, HYBRID ou SEMANTIC, de acordo com o seu caso de uso.
Aqui está o [blog](#https://aws.amazon.com/blogs/machine-learning/knowledge-bases-for-amazon-bedrock-now-supports-hybrid-search/) sobre a feature de Hybrid Search, para mais detalhes.

<div class="alert alert-block alert-warning">
<b>Nota:</b> Lembre-se de excluir a KB, o índice OSS e as roles e policies IAM relacionadas para evitar a cobrança de custos.
</div>